In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import pandas as pd
import numpy as np
import gc

paths = [
    '/kaggle/input/notebooks/fati22/embedding-titles-phase1/final_chunk_1.parquet',
    '/kaggle/input/notebooks/fati22/embedding-titles-phase1/final_chunk_2.parquet',
    '/kaggle/input/notebooks/fati22/embedding-titles-phase2/final_chunk_3.parquet',
    '/kaggle/input/notebooks/fati22/embedding-titles-phase2/final_chunk_4.parquet',
    '/kaggle/input/notebooks/fati22/embedding-titles-phase3/final_chunk_5.parquet',
    '/kaggle/input/notebooks/fati22/embedding-titles-phase3/final_chunk_6.parquet',
    '/kaggle/input/notebooks/fati22/embedding-titles-phase4/final_chunk_7.parquet'
]


In [ ]:
d_img  = 768
d_txt  = 1024
M_hnsw = 32
ef     = 200

def create_index(dim):
    core_index = faiss.IndexHNSWFlat(dim, M_hnsw)
    core_index.hnsw.efConstruction = ef
    final_index = faiss.IndexIDMap2(core_index)
    return final_index

In [ ]:
index_img = create_index(d_img)
#index_txt = create_index(d_txt)

In [ ]:
for file_path in paths:
    df      = pd.read_parquet(file_path)
    ids     = df['ID_Product'].values.astype('int64')
    img_vecs = np.vstack(df['Image_Embedding'].values).astype('float32')
    #txt_vecs = np.vstack(df['Judul_Embedding'].values).astype('float32')
    index_img.add_with_ids(img_vecs, ids)
    #index_txt.add_with_ids(txt_vecs, ids)
    del df, ids, img_vecs#,txt_vecs
    gc.collect()
faiss.write_index(index_img, "tokopedia_img_hnsw.faiss")
#faiss.write_index(index_txt, "tokopedia_txt_hnsw.faiss")